In [ ]:
!pip install pyspark findspark -q

In [ ]:
import findspark
findspark.init()

from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("RetailSalesETLProject").getOrCreate()

print("Spark Version:", spark.version)

Spark Version: 4.0.2


In [ ]:
from google.colab import files

uploaded = files.upload()

Saving Online Retail.csv to Online Retail.csv


In [ ]:
df = spark.read.csv("Online Retail.csv",header=True,inferSchema=True)

print("SCHEMA :")
df.printSchema()

print("SAMPLE DATA :")
df.show(5)

SCHEMA :
root
 |-- InvoiceNo: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- InvoiceDate: string (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- CustomerID: integer (nullable = true)
 |-- Country: string (nullable = true)

SAMPLE DATA :
+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+
|InvoiceNo|StockCode|         Description|Quantity|        InvoiceDate|UnitPrice|CustomerID|       Country|
+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+
|   536365|   85123A|WHITE HANGING HEA...|       6|12/01/2010 08:26:00|     2.55|     17850|United Kingdom|
|   536365|    71053| WHITE METAL LANTERN|       6|12/01/2010 08:26:00|     3.39|     17850|United Kingdom|
|   536365|   84406B|CREAM CUPID HEART...|       8|12/01/2010 08:26:00|     2.75|     17850|United 

In [ ]:
from pyspark.sql.functions import col

df = df.na.drop(subset=["CustomerID"])

df = df.filter(col("Quantity") > 0)

df = df.filter(col("UnitPrice") > 0)

df = df.dropDuplicates()

print("CLEANED DATA COUNT :")
print(df.count())

CLEANED DATA COUNT :
392692


In [ ]:
df = df.withColumn("TotalAmount",col("Quantity") * col("UnitPrice"))

print("FEATURE ENGINEERING :")

df.show(5)

FEATURE ENGINEERING :
+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+------------------+
|InvoiceNo|StockCode|         Description|Quantity|        InvoiceDate|UnitPrice|CustomerID|       Country|       TotalAmount|
+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+------------------+
|   536395|    22866|HAND WARMER SCOTT...|      48|12/01/2010 10:47:00|      2.1|     13767|United Kingdom|100.80000000000001|
|   536396|    82486|WOOD S/3 CABINET ...|       4|12/01/2010 10:51:00|     6.95|     17850|United Kingdom|              27.8|
|   536408|    21587|COSY HOUR GIANT T...|      12|12/01/2010 11:41:00|     2.55|     14307|United Kingdom|30.599999999999998|
|   536464|    21815|STAR  T-LIGHT HOL...|       4|12/01/2010 12:23:00|     1.45|     17968|United Kingdom|               5.8|
|   536514|    22865|HAND WARMER OWL D...|      36|12/01/2010 12:40:00|      2.1|     179

In [ ]:
df.cache()

print("DataFrame Cached Successfully")

DataFrame Cached Successfully


In [ ]:
from pyspark.sql.functions import sum

print("COUNTRY-WISE SALES :")

country_sales = df.groupBy("Country").agg(sum("TotalAmount").alias("TotalSales")).orderBy(col("TotalSales").desc())

country_sales.show(10)

COUNTRY-WISE SALES :
+--------------+------------------+
|       Country|        TotalSales|
+--------------+------------------+
|United Kingdom| 7285024.644000007|
|   Netherlands| 285446.3400000001|
|          EIRE|         265262.46|
|       Germany|228678.40000000002|
|        France|208934.31000000006|
|     Australia|138453.80999999997|
|         Spain| 61558.55999999999|
|   Switzerland|56443.949999999975|
|       Belgium| 41196.33999999996|
|        Sweden|38367.829999999994|
+--------------+------------------+
only showing top 10 rows


In [ ]:
print("TOP SELLING PRODUCTS :")

top_products = df.groupBy("Description").agg(sum("Quantity").alias("TotalQuantity")).orderBy(col("TotalQuantity").desc())

top_products.show(10, truncate=False)

TOP SELLING PRODUCTS :
+----------------------------------+-------------+
|Description                       |TotalQuantity|
+----------------------------------+-------------+
|PAPER CRAFT , LITTLE BIRDIE       |80995        |
|MEDIUM CERAMIC TOP STORAGE JAR    |77916        |
|WORLD WAR 2 GLIDERS ASSTD DESIGNS |54319        |
|JUMBO BAG RED RETROSPOT           |46078        |
|WHITE HANGING HEART T-LIGHT HOLDER|36706        |
|ASSORTED COLOUR BIRD ORNAMENT     |35263        |
|PACK OF 72 RETROSPOT CAKE CASES   |33670        |
|POPCORN HOLDER                    |30919        |
|RABBIT NIGHT LIGHT                |27153        |
|MINI PAINT SET VINTAGE            |26076        |
+----------------------------------+-------------+
only showing top 10 rows


In [ ]:
print("TOP CUSTOMERS :")

top_customers = df.groupBy("CustomerID").agg(sum("TotalAmount").alias("TotalSpent")).orderBy(col("TotalSpent").desc())

top_customers.show(10)

TOP CUSTOMERS :
+----------+------------------+
|CustomerID|        TotalSpent|
+----------+------------------+
|     14646| 280206.0200000001|
|     18102|          259657.3|
|     17450|         194390.79|
|     16446|          168472.5|
|     14911|143711.17000000013|
|     12415|124914.53000000001|
|     14156|117210.07999999996|
|     17511| 91062.38000000005|
|     16029| 80850.84000000003|
|     12346|           77183.6|
+----------+------------------+
only showing top 10 rows


In [ ]:
from pyspark.sql.window import Window
from pyspark.sql.functions import dense_rank

print("CUSTOMER RANKING :")

windowSpec = Window.orderBy(col("TotalSpent").desc())

ranked_customers = top_customers.withColumn("Rank",dense_rank().over(windowSpec))

ranked_customers.show(10)

CUSTOMER RANKING :
+----------+------------------+----+
|CustomerID|        TotalSpent|Rank|
+----------+------------------+----+
|     14646| 280206.0200000001|   1|
|     18102|          259657.3|   2|
|     17450|         194390.79|   3|
|     16446|          168472.5|   4|
|     14911|143711.17000000013|   5|
|     12415|124914.53000000001|   6|
|     14156|117210.07999999996|   7|
|     17511| 91062.38000000005|   8|
|     16029| 80850.84000000003|   9|
|     12346|           77183.6|  10|
+----------+------------------+----+
only showing top 10 rows


In [ ]:
print("SPARK SQL :")

df.createOrReplaceTempView("retail_sales")

query1 = spark.sql("""
SELECT Country,
       ROUND(SUM(TotalAmount), 2) AS Revenue
FROM retail_sales
GROUP BY Country
ORDER BY Revenue DESC
""")

query1.show(10)

SPARK SQL :
+--------------+----------+
|       Country|   Revenue|
+--------------+----------+
|United Kingdom|7285024.64|
|   Netherlands| 285446.34|
|          EIRE| 265262.46|
|       Germany|  228678.4|
|        France| 208934.31|
|     Australia| 138453.81|
|         Spain|  61558.56|
|   Switzerland|  56443.95|
|       Belgium|  41196.34|
|        Sweden|  38367.83|
+--------------+----------+
only showing top 10 rows


In [ ]:
print("PARTITIONING :")

print("Partitions Before:", df.rdd.getNumPartitions())

df = df.repartition(4)

print("Partitions After:", df.rdd.getNumPartitions())

PARTITIONING :
Partitions Before: 200
Partitions After: 4


In [ ]:
country_sales.write.mode("overwrite").csv("/content/country_sales_output",header=True)

top_products.write.mode("overwrite").csv("/content/top_products_output",header=True)

print("Output Files Saved Successfully")

Output Files Saved Successfully


In [ ]:
spark.stop()